# 1. 长期记忆 用户特定或应用级别的数据，任何会话都可以随时访问

## 1.1 记忆3种分类
Memory Type             存什么

Semantic（语义记忆）      事实

Episodic（情景记忆）      经验

Procedural（程序性记忆）   规则/做事方法

类型1：Semantic Memory（语义记忆）

即“事实类记忆”，记录事实/用户偏好/概念，如：

用户喜欢简洁回答

用户常用中文

某个公司属于哪个行业


类型2：Episodic Memory（情景记忆）

即“经验类记忆”，记录Agent过去执行的动作，如：

过去某个任务是怎么成功的

某种用户输入下，怎样回答效果最好

在Agent里，这常常表现为 few-shot examples（少样本示例）：

不直接告诉模型规则，而是给它看几个“输入 -> 输出”的例子，让它照着学

类型3：Procedural Memory（程序性记忆）

即“规则/做事方法”，如

Agent的系统提示词

Agent的工作流程

工具调用规则

## 1.1 记忆存储架构
长期记忆存储是 store---- namespace------key-------value 的四层架构

#### 第1层：Store（记忆仓库）
Store是 langgraph.store.base.BaseStore 的子类实例，由全类名可知，store是由LangGraph提供的。

常用实现类：

InMemoryStore ：将长期记忆存储在内存，适合测试

PostgresStore ：将长期记忆存储在外部的PostgreSQL数据库，适合生产环境开发期可用 InMemoryStore；生产建议数据库后端，如 PostgresStore

In [3]:
from key_value.aio.stores.memory import MemoryStore
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.postgres import PostgresSaver
from rich import print as rprint

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)

# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")

# 定义记忆管理对象
checkpointer = InMemorySaver()

model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)

# 设置线程ID
config = {
    "configurable":{
        "thread_id":"1"
    }
}
messages = [
    HumanMessage("你好啊，我是老王，你是谁？"),

]

# 创建agent
agent = create_agent(
    model=model,
    # 添加内存管理
    checkpointer=checkpointer,
)

# 第一轮对话
res = agent.invoke({
    "messages": messages,
},
config=config,
)
#
# for check in res["messages"]:
#     check.pretty_print()

    # 第2轮对话
res1 = agent.invoke({
    "messages": [HumanMessage(content="我是谁呀")]
},
    # 保证同一个线程
config=config,
)

for check in res1["messages"]:
    check.pretty_print()


================================ Human Message =================================

你好啊，我是老王，你是谁？
================================== Ai Message ==================================

你好老王！我是DeepSeek，由深度求索公司创造的AI助手。很高兴认识你！

我可以帮你解答问题、聊天、处理文件、搜索信息等等。有什么我能帮到你的吗？无论是生活琐事、工作学习，还是随便聊聊天，我都很乐意奉陪！😊
================================ Human Message =================================

我是谁呀
================================== Ai Message ==================================

哈哈，老王，你这是在考验我的记忆力吗？😄

从咱们刚才的对话来看，你是一位**自称“老王”的朋友**——至于真实姓名、职业、年龄这些，我可就不知道啦！毕竟咱们才刚认识，你还没给我透露更多信息呢。

如果你想让我更了解你，随时可以告诉我你的兴趣爱好、最近在忙什么，或者有什么特别想聊的话题。我会认真记着，下次聊天就能更投缘啦！

所以，除了“老王”这个称呼，你愿意再多分享点关于你自己的故事吗？🤔


In [4]:
from rich import print as rprint
latest_state = agent.get_state(config)
rprint(latest_state)

StateSnapshot(
    values={
        'messages': [
            HumanMessage(
                content='你好啊，我是老王，你是谁？',
                additional_kwargs={},
                response_metadata={},
                id='3571b306-4756-4a9a-a427-5acf9d1cd04d'
            ),
            AIMessage(
                content='你好老王！我是DeepSeek，由深度求索公司创造的AI助手。很高兴认识你！\n\n我可以帮你解答问题、
聊天、处理文件、搜索信息等等。有什么我能帮到你的吗？无论是生活琐事、工作学习，还是随便聊聊天，我都很乐意奉陪！😊',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 63,
                        'prompt_tokens': 12,
                        'total_tokens': 75,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0
                        },
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 12
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                    'id': '94f6d96d-60f4-45dd-b56c-e1d1d6f4118a',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a01805-4033-7e81-97c2-336a8655628c-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 12,
                    'output_tokens': 63,
                    'total_tokens': 75,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            ),
            HumanMessage(
                content='我是谁呀',
                additional_kwargs={},
                response_metadata={},
                id='5ac03127-c7a7-41d6-a6fc-cc28a4ab9faf'
            ),
            AIMessage(
                content='哈哈，老王，你这是在考验我的记忆力吗？😄\n\n从咱们刚才的对话来看，你是一位**自称“老王”的朋
友**——至于真实姓名、职业、年龄这些，我可就不知道啦！毕竟咱们才刚认识，你还没给我透露更多信息呢。\n\n如果你想让我更
了解你，随时可以告诉我你的兴趣爱好、最近在忙什么，或者有什么特别想聊的话题。我会认真记着，下次聊天就能更投缘啦！\n\
n所以，除了“老王”这个称呼，你愿意再多分享点关于你自己的故事吗？🤔',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 117,
                        'prompt_tokens': 82,
                        'total_tokens': 199,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0
                        },
                        'prompt_cache_hit_tokens': 0,
                        'prompt_cache_miss_tokens': 82
                    },
                    'model_provider': 'deepseek',
                    'model_name': 'deepseek-v4-flash',
                    'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                    'id': 'da4662b6-c0c9-4550-a43e-e706ba804f54',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a01805-4854-7ff0-b828-1a04496b8c8b-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 82,
                    'output_tokens': 117,
                    'total_tokens': 199,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {}
                }
            )
        ]
    },
    next=(),
    config={
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f1

InMemorySaver会丢失数据吗？

会，它只保存在内存中

同一进程有效

程序重启会丢失数据

不同进程无法共享

解决方案：持久化（SQLite、PostgreSQL）

内存会无限增长吗？
会，默认情况下，它会保存所有消息

问题：

消息越来越多（无限增长、需要管理上下文）

token消耗增加、甚至超过模型token限制

响应速度变慢、成本增加

解决方案：上下文管理（修剪、摘要）

如何清空某个回话的历史？

目前In MemorySaver没有提供删除API

临时解决方案：使用新的Thread_i、或重新创建Agent

#### 第2层：Namespace（命名空间）
数据类型是由任意长度的 tuple[str, ...] 表示的 层级路径 。作用上很像“文件路径 / 文件夹层级”，用于
给长期记忆分组和隔离。数据类型为 字符串元组

#### 第3层：Key（键）
是该 namespace 下的唯一标识，单条记忆的唯一键，数据类型为 字符串(str)

#### 第4层：Value（值）
是存储的值，数据类型为 字典(dict[str, Any])

In [6]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
namespace = ("users",)
user_id = 'user-1'
username = "小蓝"
store.put(namespace, user_id, {"name": username})
print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '小蓝'}, created_at='2026-08-19T07:23:00.849151+00:00', updated_at='2026-08-19T07:23:00.849153+00:00')


更新

In [8]:
store.put(namespace, user_id, {"name": "效率"})
print(store.get(namespace, user_id))

Item(namespace=['users'], key='user-1', value={'name': '效率'}, created_at='2026-08-19T07:23:41.176498+00:00', updated_at='2026-08-19T07:23:41.176499+00:00')


### 1.3 基于search() 检索AIP

In [20]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
"course": "计算机组成原理",
"sports": "跑步",
"food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
"course": "数字电路与模拟电路",
"sports": "跑步",
"food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
"course": "数字电路与模拟电路",
"sports": "羽毛球",
"food": "紫光园奶皮子酸奶"
}
store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)


按照namespace检索

In [19]:
for item in store.search(("users",)):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-19T07:32:01.022302+00:00', updated_at='2026-08-19T07:32:01.022304+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-08-19T07:32:01.022323+00:00', updated_at='2026-08-19T07:32:01.022324+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-19T07:32:01.022338+00:00', updated_at='2026-08-19T07:32:01.022338+00:00', score=None)


按照filter过滤

In [26]:
for item in store.search(("users",),filter={"sports":"羽毛球"}):
    print(item)

Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-08-19T07:32:41.770079+00:00', updated_at='2026-08-19T07:32:41.770080+00:00', score=None)


## 1.3 在Agent运行中访问长期记忆

### 1.3.1 在工具中访问长期记忆
runtime: ToolRuntime 对象可以获取store对象，并操作其对象

In [33]:

from langchain_core.tools import tool
from typing import NotRequired
from langgraph.prebuilt.tool_node import ToolRuntime
from langchain_core.messages import HumanMessage

from langchain.agents import create_agent, AgentState
from langgraph.store.memory import InMemoryStore
import os

from dotenv import load_dotenv

from langchain.agents.factory import create_agent,init_chat_model

# 加载配置文件，存在相同key采用当前覆盖
load_dotenv(override=True)


# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")


model = init_chat_model(
    model=DEEPSEEK_MODEL_NAME,
    model_provider="deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_API_BASE,
    extra_body={"thinking":{"type":"disabled"}}

)
# 具体模型的key和url
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE   = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL_NAME   = os.getenv("DEEPSEEK_MODEL")


store = InMemoryStore()
class CustomState(AgentState):
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """
    将用户信息保存在长期记忆中

    Args:
    name: 用户名

    Returns:
    str: 保存状态
    """
    runtime.store.put(("users",), runtime.state["user_id"], {"name": name})
    return "saved"


@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """
    从长期记忆中读取用户信息

    Returns:
    str: 用户信息
    """
    item = runtime.store.get(("users",), runtime.state["user_id"])
    return str(item.value) if item else "unknown"
    agent = create_agent(
    model=model,
    tools=[save_user_info, get_user_info],
    store=store,
    system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
    state_schema=CustomState,
    )


# 创建agent
agent = create_agent(
model=model,
tools=[save_user_info, get_user_info],
store=store,
system_prompt="用户提及个人信息时及时记录，用户询问个人信息时尝试用工具检索",
state_schema=CustomState,
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
"messages": [HumanMessage("你好，很高兴认识你，我是小花")],
"user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()
print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
"messages": [HumanMessage("我是谁")],
"user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花！很高兴认识你！😊

让我把你的名字记下来，这样以后我们聊天的时候我就能记得你了。
Tool Calls:
  save_user_info (call_00_MHMurzdevybKXOsSG8Y60157)
 Call ID: call_00_MHMurzdevybKXOsSG8Y60157
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

已经记住你的名字啦，小花！以后我们聊天时会更加亲切。有什么我可以帮你的吗？无论是聊天、问问题还是分享心情，我都很乐意陪着你！✨
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================

让我先查看一下长期记忆中保存的用户信息。
Tool Calls:
  get_user_info (call_00_lFtx4qlo7

### 1.4 何时写入记忆

1. 在主流程里写（hot path）
也就是：用户发消息，AI 一边回答，一边决定要不要记下来。

优点：
立即生效
下一轮马上能用
用户可感知，透明

缺点：
增加延迟
逻辑变复杂

2. 在后台写（background）
就是先回答用户，记忆整理放到后台 异步 做。

优点：
主流程更快
记忆逻辑更独立
更适合批量整理

缺点：
不能立刻生效
要决定多久整理一次
触发时机不好选


工程上通常这么选：

用户偏好、账号资料 ：可热路径写

对话摘要、经验沉淀、行为分析 ：更适合后台写